### Read the Data

In [29]:
import json 
import pandas as pd
df = pd.read_json('data/python_train_0.jsonl', orient='records', lines=True)
df.head()
df.index

RangeIndex(start=0, stop=30000, step=1)

In [6]:
df.columns


Index(['repo', 'path', 'func_name', 'original_string', 'language', 'code',
       'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url',
       'partition'],
      dtype='object')

### Load the base slm model

In [2]:
# pip install transformers
from transformers import AutoModelForCausalLM, AutoTokenizer
checkpoint = "HuggingFaceTB/SmolLM-135M"
device = "mps" # for GPU usage or "cpu" for CPU usage
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
# for multiple GPUs install accelerate and do `model = AutoModelForCausalLM.from_pretrained(checkpoint, device_map="auto")`
model = AutoModelForCausalLM.from_pretrained(checkpoint).to(device)
inputs = tokenizer.encode("def fibonnaci(n):", return_tensors="pt").to(device)
outputs = model.generate(inputs)
print(tokenizer.decode(outputs[0]))


/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


def fibonnaci(n):
    if n < 2:
        return n
    return fibonnaci(n-1


In [3]:
print(f"Memory footprint: {model.get_memory_footprint() / 1e6:.2f} MB")


Memory footprint: 538.06 MB


### Load the required libraries

In [30]:
# pip install transformers trl datasets accelerate wandb
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset, Dataset
from trl import SFTTrainer, SFTConfig
import wandb
from datasets import Dataset
from transformers import TrainingArguments

##### Setup wandb to log our model adapters and metrics

In [3]:
wandb.login()
wandb.init(project="smollm-codesearchnet", name="smollm135m-finetune")


wandb: Currently logged in as: lordsahu (lordsahu-publicis-sapient) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")


#### Converting the data into a Hugging Face dataset to use it in its environment

In [31]:
dataset = Dataset.from_pandas(df)
dataset

RangeIndex(start=0, stop=30000, step=1)


Dataset({
    features: ['repo', 'path', 'func_name', 'original_string', 'language', 'code', 'code_tokens', 'docstring', 'docstring_tokens', 'sha', 'url', 'partition'],
    num_rows: 30000
})

### Loading the pre trained tokenizer for the slm model

In [32]:
tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM-135M")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token 

#### Constructing the output format

In [33]:
def format_fn(example):
    """
    Combine the 'docstring' and 'code' fields into one prompt+completion string:

      1. '### Docstring:' marker + docstring text
      2. Blank line
      3. '### Code:' marker + full code text

    Args:
        example (dict): one row from the dataset,
          with keys 'docstring' and 'code'.

    Returns:
        str: formatted text to feed into the model.
    """
    prompt = example["docstring"].strip()
    code    = example["code"].strip()
    return (
        "### Docstring:\n"
        f"{prompt}\n\n"
        "### Code:\n"
        f"{code}"
    )

#### Mapping and tokenising the dataset to fetch the required columns

In [34]:
# Map format_fn over the dataset, dropping all other columns
dataset_formatted = dataset.map(
    lambda ex: {"text": format_fn(ex)},
    remove_columns=dataset.column_names,
)

Map: 100%|██████████| 30000/30000 [00:00<00:00, 36886.44 examples/s]


In [ ]:
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=False,    
        padding="longest",
    )

tokenized_dataset = dataset_formatted.map(
    tokenize_fn,
    batched=True,
)

Map: 100%|██████████| 30000/30000 [00:30<00:00, 968.87 examples/s] 


#### Creating the train test split 

In [36]:
split_dataset = tokenized_dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

### Running the SFT trainer without PEFT, i.e on full weights

In [37]:

# Define SFTConfig (replaces TrainingArguments)
sft_config = SFTConfig(
    output_dir="./smollm-sft-output",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_dir="./logs",
    logging_steps=50,
    report_to="wandb", 
    learning_rate=2e-5,
    warmup_steps=100,
    weight_decay=0.01,
    max_seq_length=None,  
    dataset_text_field="text", 
    push_to_hub=False
)


#### Running the SFT config on the slm

In [14]:
# Load the base model
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM-135M",
    trust_remote_code=True,
)

# Define SFTTrainer (Supervised Fine-Tuning)
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=sft_config,
    processing_class=tokenizer,
)

Truncating eval dataset: 100%|██████████| 1/1 [00:00<00:00, 540.85 examples/s]


In [39]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,10.445983
2,No log,10.293784
3,No log,10.054584


/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=9, training_loss=8.018429226345486, metrics={'train_runtime': 967.6816, 'train_samples_per_second': 0.028, 'train_steps_per_second': 0.009, 'total_flos': 17617878908928.0, 'train_loss': 8.018429226345486})

In [ ]:
# Evaluate perplexity or loss on validation set
eval_metrics = trainer.evaluate()
print(eval_metrics)

# Generate from a few validation examples
from random import sample

# Pick a random validation examples
samples = sample(list(val_dataset),1)

# Decode model predictions
for i, sample_input in enumerate(samples):
    input_ids = tokenizer(sample_input["text"], return_tensors="pt").input_ids.to(model.device)
    output_ids = model.generate(input_ids, max_new_tokens=100, do_sample=True, temperature=0.7)
    
    print(f"\n🧪 Example {i+1}:")
    print("="*40)
    print("Prompt:\n", sample_input["text"][:400], "...\n")
    print("Generated Code:\n", tokenizer.decode(output_ids[0], skip_special_tokens=True)[len(sample_input["text"]):])
    print("="*40)


/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


{'eval_loss': 10.054583549499512, 'eval_runtime': 1.8745, 'eval_samples_per_second': 0.533, 'eval_steps_per_second': 0.533}

🧪 Example 1:
Prompt:
 ### Docstring:
Execute the KEYS command on all Redis shards.

        Args:
            pattern: The KEYS pattern to query.

        Returns:
            The concatenated list of results from all shards.

### Code:
def _keys(self, pattern):
        """Execute the KEYS command on all Redis shards.

        Args:
            pattern: The KEYS pattern to query.

        Returns:
            The conca ...

Generated Code:
 

### Code:
def _keys_single(self, pattern):
    """Execute the KEYS command on all Redis shards.

        Args:
            pattern: The KEYS pattern to query.

        Returns:
            The list of results from all shards.

### Code:
def _keys_single(self, pattern):
    """Execute the KEYS command on all Redis shards.

        Args:
            pattern: The KEYS pattern to query


## Results: 
#### Bad code generation as updating the whole weights might not be the best idea, as I am using a small subset of data to train my slm which works negatively as the slm is pretrained on huge chunks of coding data 

In [ ]:
# Save locally
# trainer.save_model("./smollm-finetuned-codegen")

# # Optional: Push to Hugging Face Hub
# model.push_to_hub("lordsahu/smollm-codesearchnet-finetuned_v1")
# tokenizer.push_to_hub("lordsahu/smollm-codesearchnet-finetuned_v1")

### Using the SFT with LORA configurations
#### This will allows us to finetune a percentage of weights 
#### Creating a LORA parameters list 

In [38]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                          
    lora_alpha=16,               
    lora_dropout=0.05,           
    bias="none",                  
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"], 
)

#### Calling the base model 

In [39]:
model = AutoModelForCausalLM.from_pretrained(
    "HuggingFaceTB/SmolLM-135M",
    trust_remote_code=True,
    load_in_8bit=False
)  

In [40]:
peft_model = get_peft_model(model, lora_config)
print(f"Trainable params: {peft_model.print_trainable_parameters()}")


trainable params: 921,600 || all params: 135,436,608 || trainable%: 0.6805
Trainable params: None


#### Creating the SFT trainer, now it will be using the LORA PEFT model

In [41]:
training_args = SFTConfig(
    output_dir="./smollm-sft-lora-output",
    per_device_train_batch_size=8,    
    gradient_accumulation_steps=4,    
    learning_rate=2e-4,               
    num_train_epochs=3,
    logging_steps=1,                  
    eval_strategy="steps",      
    eval_steps=2,                     
    save_strategy="steps",
    save_steps=5,
    fp16=False,                    
    max_seq_length=512,               
    dataset_text_field="text",
    warmup_ratio=0.03,                
    weight_decay=0.01,
    report_to="wandb"                 
)

In [42]:
dataset = Dataset.from_pandas(df.reset_index(drop=True))


#### Calling the SFT Trainer class to run the finetuning model

In [44]:
trainer = SFTTrainer(
    model=peft_model,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    args=training_args,
    processing_class=tokenizer,
)

Truncating train dataset:   0%|          | 0/27000 [00:00<?, ? examples/s]

Truncating eval dataset: 100%|██████████| 3000/3000 [00:00<00:00, 70702.83 examples/s]
No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [ ]:
trainer.train()

/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Step,Training Loss,Validation Loss
2,6.438800,6.343796
4,5.351200,6.340252
6,7.189300,6.333813
8,6.088300,6.324357
10,6.066100,6.311684
12,5.905200,6.295543
14,6.385800,6.275568
16,6.529500,6.251340
18,6.969800,6.222355
20,6.023600,6.188076


/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory w

In [ ]:
from transformers import AutoTokenizer
from peft import PeftModel, AutoPeftModelForCausalLM
import torch

model_path = 'smollm-sft-lora-output/checkpoint-140'
repo_name = 'LordSahu/smollm-codesearchnet-sft-peft-finetuned_v1'

print('Loading model...')
try:
    model = AutoPeftModelForCausalLM.from_pretrained(model_path, torch_dtype=torch.float16)
    tokenizer = AutoTokenizer.from_pretrained('HuggingFaceTB/SmolLM-135M')
    print('Model loaded successfully!')
    

    print('Pushing model to', repo_name)
    model.push_to_hub(repo_name, commit_message='Uploading fine-tuned SmolLM code generation model')
    tokenizer.push_to_hub(repo_name, commit_message='Uploading tokenizer')
    print('Done!')
except Exception as e:
    print('Error:', e)


In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

model_id = "LordSahu/smollm-codesearchnet-sft-peft-finetuned_v1"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

pipe = pipeline("text-generation", model=model, tokenizer=tokenizer)
prompt = "Write a function to calculate the factorial of a number\n\n"
result = pipe(prompt, max_new_tokens=400, temperature=0.1, top_p=0.95)
print(result[0]['generated_text'])

/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Device set to use mps:0
/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:631: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/Users/utksahu2/Desktop/slm-code-finetune/.venv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:636: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.95` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
Setting `pad_to

Write a function to calculate the factorial of a number


def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n-1)


#### The results will be shown on the streamlit app